# Module 06: Adaptive Card Designer

## Learning Objectives

By completing this notebook, you will:
1. Build Adaptive Card JSON programmatically in Python
2. Create reusable card component builders for common approval patterns
3. Validate card JSON against the Adaptive Card schema structure
4. Generate different card types: approval, status update, data collection
5. Inject dynamic data into card templates using Python string formatting

## Prerequisites

- Module 06 Guide 02: Adaptive Cards and Business Process Patterns
- Basic Python (dictionaries, lists, functions)
- No external API keys required — this notebook generates and validates JSON locally

## Estimated Time: 15 minutes

---

## Why Build Cards in Python?

Power Automate's card field accepts static JSON. When your card content is fully dynamic — generated from a database query, a list of approvers, or runtime calculations — you need a programmatic way to construct the JSON before passing it to the flow. This notebook gives you that toolkit.

Real-world uses:
- Azure Function generates a card JSON string → passes it to Power Automate via HTTP action
- Python script in a scheduled flow generates summary cards from overnight data
- Automated tests validate card structure before deploying flow changes

In [ ]:
# Setup: standard library only — no pip install required
import json
from typing import Any

# Utility: pretty-print card JSON for inspection
def show_card(card: dict, label: str = "Adaptive Card") -> None:
    """
    Print a formatted JSON representation of an Adaptive Card.

    Parameters
    ----------
    card : dict
        The card dictionary to display
    label : str
        Display label printed above the JSON
    """
    print(f"=== {label} ===")
    print(json.dumps(card, indent=2))
    print()

print("Setup complete. Ready to build Adaptive Cards.")

## 1. Card Building Blocks

Each element in an Adaptive Card's `body` array is a Python dictionary. We build helper functions for each element type. This approach:

- Enforces required fields (so we don't accidentally omit `type`)
- Provides sensible defaults (so we don't repeat common settings)
- Makes cards readable at a glance (function names describe intent)

We start with the five elements used in most approval cards.

In [ ]:
# --- Card Element Builders ---
# Each function returns a dictionary representing one Adaptive Card body element.
# The returned dict is placed directly into the card's body[] or items[] array.

def text_block(
    text: str,
    size: str = "Default",
    weight: str = "Default",
    color: str = "Default",
    wrap: bool = True,
    spacing: str = "Default"
) -> dict:
    """
    Build a TextBlock element.

    Parameters
    ----------
    text : str
        Display text (supports limited Markdown: **bold**, *italic*)
    size : str
        Font size: Small | Default | Medium | Large | ExtraLarge
    weight : str
        Font weight: Lighter | Default | Bolder
    color : str
        Color: Default | Dark | Light | Accent | Good | Warning | Attention
    wrap : bool
        Whether to wrap text at container boundary
    spacing : str
        Top spacing: None | Small | Default | Medium | Large | ExtraLarge | Padding
    """
    return {
        "type": "TextBlock",
        "text": text,
        "size": size,
        "weight": weight,
        "color": color,
        "wrap": wrap,
        "spacing": spacing
    }


def fact_set(facts: dict[str, str]) -> dict:
    """
    Build a FactSet element from a dictionary of label-value pairs.

    Parameters
    ----------
    facts : dict[str, str]
        Keys become fact titles (labels); values become fact values.
        Insertion order is preserved (Python 3.7+).

    Example
    -------
    >>> fact_set({"Amount:": "$450", "Requested by:": "Priya"})
    """
    return {
        "type": "FactSet",
        "facts": [
            {"title": str(k), "value": str(v)}
            for k, v in facts.items()
        ]
    }


def container(
    items: list,
    style: str = "default",
    spacing: str = "Default"
) -> dict:
    """
    Build a Container element wrapping a list of body items.

    Parameters
    ----------
    items : list
        List of element dicts to place inside the container
    style : str
        Visual style: default | emphasis | good | attention | warning | accent
    spacing : str
        Top spacing before this container
    """
    return {
        "type": "Container",
        "style": style,
        "spacing": spacing,
        "items": items
    }


def input_text(
    input_id: str,
    label: str,
    placeholder: str = "",
    is_multiline: bool = False,
    is_required: bool = False
) -> dict:
    """
    Build an Input.Text element for collecting freeform text.

    Parameters
    ----------
    input_id : str
        Identifier used to read the value from the response body.
        Must be unique within the card.
    label : str
        Display label shown above the text field
    placeholder : str
        Grey hint text shown when the field is empty
    is_multiline : bool
        If True, renders as a textarea rather than a single line
    is_required : bool
        If True, card will not submit until this field is filled
    """
    return {
        "type": "Input.Text",
        "id": input_id,
        "label": label,
        "placeholder": placeholder,
        "isMultiline": is_multiline,
        "isRequired": is_required
    }


def input_choice_set(
    input_id: str,
    label: str,
    choices: list[tuple[str, str]],
    style: str = "expanded",
    is_required: bool = True
) -> dict:
    """
    Build an Input.ChoiceSet element (radio buttons or dropdown).

    Parameters
    ----------
    input_id : str
        Identifier for reading the selected value from the response body.
    label : str
        Display label shown above the choices
    choices : list[tuple[str, str]]
        List of (display_title, submitted_value) pairs.
        The submitted_value is what Power Automate receives.
    style : str
        Rendering style: expanded (radio buttons) | compact (dropdown)
    is_required : bool
        If True, a choice must be selected before the card can be submitted.
        Always set to True for decision inputs.

    Example
    -------
    >>> input_choice_set("decision", "Your Decision",
    ...     [("Approve", "approve"), ("Reject", "reject")])
    """
    return {
        "type": "Input.ChoiceSet",
        "id": input_id,
        "label": label,
        "style": style,
        "isRequired": is_required,
        "choices": [
            {"title": title, "value": value}
            for title, value in choices
        ]
    }


def action_submit(title: str, style: str = "positive") -> dict:
    """
    Build an Action.Submit element (submit button).

    Parameters
    ----------
    title : str
        Button label text
    style : str
        Button color: positive (blue) | destructive (red) | default
    """
    return {
        "type": "Action.Submit",
        "title": title,
        "style": style
    }


print("Element builders defined.")
print("Available builders: text_block, fact_set, container, input_text, input_choice_set, action_submit")

## 2. Card Factory Function

The `make_card` function assembles the top-level card object. Every Adaptive Card requires the same outer shell: `type`, `version`, `body`, and `actions`. This function ensures you never forget those required fields.

In [ ]:
# Assemble body elements and actions into a complete card dictionary.
# Version 1.4 is the correct target for Microsoft Teams and Outlook.

def make_card(body: list, actions: list, version: str = "1.4") -> dict:
    """
    Assemble a complete Adaptive Card dictionary.

    Parameters
    ----------
    body : list
        List of element dicts (TextBlock, FactSet, Container, Input.*, etc.)
    actions : list
        List of Action dicts (Action.Submit, Action.OpenUrl, etc.)
    version : str
        Adaptive Cards schema version. Teams supports up to 1.4.
        Do not use 1.5+ unless you have confirmed host support.

    Returns
    -------
    dict
        Valid Adaptive Card top-level object.
        Pass json.dumps(card) as the Message field in Power Automate.
    """
    return {
        "type": "AdaptiveCard",
        "version": version,
        "body": body,
        "actions": actions
    }


# Build and display a minimal test card to verify the function works
test_card = make_card(
    body=[
        text_block("Hello from Python!", size="Large", weight="Bolder")
    ],
    actions=[
        action_submit("OK")
    ]
)

show_card(test_card, "Minimal Test Card")

## 3. Approval Card Template

This function generates a standard single-stage approval card. It accepts structured request data and produces a card ready to paste into Power Automate (or inject via HTTP).

The card includes:
- Header with request title
- FactSet with request details
- Emphasized container for justification text
- Decision choice set (Approve / Reject)
- Optional comments text input
- Submit button

In [ ]:
# Build a reusable approval card template.
# Separating the template function from the data that fills it
# lets you generate many cards with one function.

def build_approval_card(
    title: str,
    request_facts: dict[str, str],
    justification: str,
    decision_choices: list[tuple[str, str]] = None,
    include_comments: bool = True,
    comments_label: str = "Comments (optional)"
) -> dict:
    """
    Generate an approval Adaptive Card with dynamic data.

    Parameters
    ----------
    title : str
        Card header text, e.g. "Expense Approval Required"
    request_facts : dict[str, str]
        Ordered dict of label-value pairs shown in the FactSet.
        e.g. {"Requested by:": "Priya Patel", "Amount:": "$450.00"}
    justification : str
        Business justification text shown in the emphasis container.
    decision_choices : list[tuple[str, str]], optional
        List of (display_title, submitted_value) pairs.
        Defaults to [("Approve", "approve"), ("Reject", "reject")].
    include_comments : bool
        Whether to include a comments text input below the decision field.
    comments_label : str
        Label for the comments input field.

    Returns
    -------
    dict
        Complete Adaptive Card dictionary.

    Example
    -------
    >>> card = build_approval_card(
    ...     title="Expense Approval Required",
    ...     request_facts={"Amount:": "$450", "Category:": "Software"},
    ...     justification="Annual IDE renewal for backend team."
    ... )
    >>> print(json.dumps(card, indent=2))
    """
    # Default to binary Approve/Reject if no custom choices provided
    if decision_choices is None:
        decision_choices = [
            ("Approve", "approve"),
            ("Reject", "reject")
        ]

    body = [
        # Card header — large, bold, accent color draws the eye immediately
        text_block(title, size="Large", weight="Bolder", color="Accent"),

        # Subtitle sets context for approver
        text_block(
            "Please review the following request and submit your decision.",
            color="Default",
            spacing="Small"
        ),

        # Structured request data in a scannable label-value layout
        fact_set(request_facts),

        # Justification in an emphasis container — visually distinct from facts
        container(
            style="emphasis",
            spacing="Medium",
            items=[
                text_block("Business Justification", weight="Bolder"),
                text_block(justification, wrap=True, spacing="Small")
            ]
        ),

        # Decision input — required, expanded = radio buttons (one click)
        input_choice_set(
            input_id="decision",
            label="Your Decision",
            choices=decision_choices,
            style="expanded",
            is_required=True
        )
    ]

    # Conditionally add comments field — optional for most approvals
    if include_comments:
        body.append(
            input_text(
                input_id="comments",
                label=comments_label,
                placeholder="Add context for the requestor...",
                is_multiline=True,
                is_required=False
            )
        )

    return make_card(
        body=body,
        actions=[action_submit("Submit Decision", style="positive")]
    )


# Generate a sample expense approval card
expense_card = build_approval_card(
    title="Expense Approval Required",
    request_facts={
        "Requested by:": "Priya Patel",
        "Department:": "Engineering",
        "Amount:": "$450.00",
        "Category:": "Software License",
        "Submitted:": "2024-03-15"
    },
    justification="Annual renewal for JetBrains IDEs used by the backend team (8 developers)."
)

show_card(expense_card, "Expense Approval Card")

## 4. Card Validation

Before using a card in production, validate it programmatically. Structural validation catches errors that would only surface at runtime in Power Automate — saving you a failed flow run and an awkward card sent to the wrong person.

We validate three layers:
1. **Top-level structure** — required fields present with correct types
2. **Body element integrity** — every element has a `type`, inputs have an `id`
3. **Input uniqueness** — no duplicate `id` values (duplicate ids silently overwrite each other)

In [ ]:
# Card validator: checks structural correctness without making any API calls.
# Returns a list of error strings (empty list = card is valid).

SUPPORTED_VERSIONS = {"1.0", "1.1", "1.2", "1.3", "1.4"}
INPUT_TYPES = {"Input.Text", "Input.Number", "Input.Date", "Input.Time", "Input.Toggle", "Input.ChoiceSet"}
VALID_BODY_TYPES = {
    "TextBlock", "Image", "Media",
    "Container", "ColumnSet", "Column",
    "FactSet", "ImageSet",
    "ActionSet", "RichTextBlock",
} | INPUT_TYPES


def validate_card(card: dict) -> list[str]:
    """
    Validate an Adaptive Card dictionary for structural correctness.

    Parameters
    ----------
    card : dict
        Adaptive Card top-level dictionary

    Returns
    -------
    list[str]
        List of error messages. Empty list means the card is valid.
    """
    errors = []

    # --- Layer 1: Top-level structure ---

    # type must be "AdaptiveCard"
    if card.get("type") != "AdaptiveCard":
        errors.append(f"Top-level 'type' must be 'AdaptiveCard', got: {card.get('type')!r}")

    # version must be present and a supported string
    version = card.get("version")
    if version is None:
        errors.append("'version' field is required")
    elif str(version) not in SUPPORTED_VERSIONS:
        errors.append(
            f"'version' {version!r} may not be supported in Teams/Outlook. "
            f"Supported: {sorted(SUPPORTED_VERSIONS)}"
        )

    # body must be present and a list
    body = card.get("body")
    if body is None:
        errors.append("'body' field is required")
    elif not isinstance(body, list):
        errors.append(f"'body' must be a list, got: {type(body).__name__}")
    elif len(body) == 0:
        errors.append("'body' is empty — card will display nothing")

    # actions must be a list if present
    actions = card.get("actions", [])
    if not isinstance(actions, list):
        errors.append(f"'actions' must be a list, got: {type(actions).__name__}")

    # --- Layer 2: Body element integrity ---
    if isinstance(body, list):
        errors.extend(_validate_elements(body, path="body"))

    # --- Layer 3: Input id uniqueness ---
    if isinstance(body, list):
        input_ids = _collect_input_ids(body)
        seen_ids = set()
        for input_id in input_ids:
            if input_id in seen_ids:
                errors.append(
                    f"Duplicate input id {input_id!r} — duplicate ids overwrite each other in the response"
                )
            seen_ids.add(input_id)

    return errors


def _validate_elements(elements: list, path: str) -> list[str]:
    """Recursively validate body or container items."""
    errors = []
    for idx, element in enumerate(elements):
        elem_path = f"{path}[{idx}]"

        if not isinstance(element, dict):
            errors.append(f"{elem_path}: element must be a dict, got {type(element).__name__}")
            continue

        elem_type = element.get("type")
        if not elem_type:
            errors.append(f"{elem_path}: missing 'type' field")
            continue

        if elem_type not in VALID_BODY_TYPES:
            errors.append(f"{elem_path}: unknown element type {elem_type!r}")

        # Input elements must have an id
        if elem_type in INPUT_TYPES and not element.get("id"):
            errors.append(f"{elem_path} ({elem_type}): 'id' is required on all input elements")

        # FactSet must have at least one fact
        if elem_type == "FactSet":
            facts = element.get("facts", [])
            if not facts:
                errors.append(f"{elem_path} (FactSet): 'facts' array is empty")

        # Recurse into Container items
        if elem_type == "Container" and "items" in element:
            errors.extend(_validate_elements(element["items"], path=f"{elem_path}.items"))

        # Recurse into ColumnSet columns
        if elem_type == "ColumnSet" and "columns" in element:
            for col_idx, col in enumerate(element["columns"]):
                if isinstance(col, dict) and "items" in col:
                    errors.extend(
                        _validate_elements(col["items"], path=f"{elem_path}.columns[{col_idx}].items")
                    )

    return errors


def _collect_input_ids(elements: list) -> list[str]:
    """Recursively collect all input ids from body elements."""
    ids = []
    for element in elements:
        if not isinstance(element, dict):
            continue
        elem_type = element.get("type", "")
        if elem_type in INPUT_TYPES and element.get("id"):
            ids.append(element["id"])
        if elem_type == "Container" and "items" in element:
            ids.extend(_collect_input_ids(element["items"]))
        if elem_type == "ColumnSet" and "columns" in element:
            for col in element["columns"]:
                if isinstance(col, dict) and "items" in col:
                    ids.extend(_collect_input_ids(col["items"]))
    return ids


def validate_and_report(card: dict, label: str = "Card") -> bool:
    """
    Validate a card and print a human-readable report.

    Returns True if valid, False if errors were found.
    """
    errors = validate_card(card)
    if errors:
        print(f"[INVALID] {label} — {len(errors)} error(s) found:")
        for error in errors:
            print(f"  - {error}")
        return False
    else:
        print(f"[VALID] {label} — no structural errors detected")
        return True


# Validate the expense card we built above
validate_and_report(expense_card, "Expense Approval Card")

## 5. Additional Card Types

Approval decisions are just one use case for Adaptive Cards in Power Automate flows. Two other common patterns are:

- **Status Update Card** — posted to a channel to announce a completed workflow (no inputs, no submit button)
- **Data Collection Card** — sent to a user to gather structured information as input to a subsequent process

These card types use the same building blocks, assembled differently.

In [ ]:
# Status Update Card — announcement card with no interactive elements.
# Used to post a summary to a Teams channel after a process completes.

def build_status_card(
    title: str,
    status: str,
    status_color: str,
    summary_facts: dict[str, str],
    message: str = ""
) -> dict:
    """
    Build a status announcement card with no interactive inputs.

    Parameters
    ----------
    title : str
        Card header
    status : str
        Status label, e.g. "Approved", "Rejected", "Completed"
    status_color : str
        Color token for the status text: Good | Attention | Warning | Accent
    summary_facts : dict[str, str]
        Summary data for the FactSet
    message : str
        Optional explanatory text below the facts
    """
    body = [
        text_block(title, size="Medium", weight="Bolder"),
        text_block(status, size="Large", weight="Bolder", color=status_color),
        fact_set(summary_facts)
    ]

    if message:
        body.append(
            container(
                style="emphasis",
                spacing="Medium",
                items=[text_block(message, wrap=True)]
            )
        )

    # No actions — this card is informational only
    return make_card(body=body, actions=[])


# Generate a status update card for an approved expense
approval_status_card = build_status_card(
    title="Expense Request Update",
    status="APPROVED",
    status_color="Good",
    summary_facts={
        "Request:": "Software License — JetBrains IDEs",
        "Amount:": "$450.00",
        "Approved by:": "Alex Chen (Engineering Manager)",
        "Approved at:": "2024-03-15 14:32 UTC"
    },
    message="Your expense request has been approved. You may proceed with the purchase. Attach the receipt to this request when complete."
)

show_card(approval_status_card, "Status Update Card (Approved)")
validate_and_report(approval_status_card, "Status Update Card")

In [ ]:
# Data Collection Card — gathers structured input from a user.
# Example: IT onboarding form sent to a new hire's manager.

def build_data_collection_card(
    title: str,
    description: str,
    fields: list[dict]
) -> dict:
    """
    Build a data collection card from a field specification list.

    Parameters
    ----------
    title : str
        Card header
    description : str
        Instructions displayed below the header
    fields : list[dict]
        List of field specifications. Each dict must have:
        - 'type': 'text' | 'choice'
        - 'id': unique string identifier
        - 'label': display label
        - 'required': bool
        For 'text' type: optional 'multiline' (bool), 'placeholder' (str)
        For 'choice' type: 'choices' list of (title, value) tuples

    Returns
    -------
    dict
        Complete Adaptive Card dictionary
    """
    body = [
        text_block(title, size="Large", weight="Bolder"),
        text_block(description, wrap=True, spacing="Small")
    ]

    for field in fields:
        field_type = field["type"]

        if field_type == "text":
            body.append(input_text(
                input_id=field["id"],
                label=field["label"],
                placeholder=field.get("placeholder", ""),
                is_multiline=field.get("multiline", False),
                is_required=field.get("required", False)
            ))

        elif field_type == "choice":
            body.append(input_choice_set(
                input_id=field["id"],
                label=field["label"],
                choices=field["choices"],
                style=field.get("style", "compact"),
                is_required=field.get("required", False)
            ))

        else:
            raise ValueError(f"Unknown field type: {field_type!r}. Use 'text' or 'choice'.")

    return make_card(
        body=body,
        actions=[action_submit("Submit", style="positive")]
    )


# IT onboarding data collection card
onboarding_card = build_data_collection_card(
    title="New Employee IT Onboarding",
    description="Please complete the following information to set up IT access for your new team member.",
    fields=[
        {
            "type": "text",
            "id": "employeeEmail",
            "label": "Employee Email Address",
            "placeholder": "firstname.lastname@contoso.com",
            "required": True
        },
        {
            "type": "choice",
            "id": "department",
            "label": "Department",
            "choices": [
                ("Engineering", "engineering"),
                ("Sales", "sales"),
                ("Finance", "finance"),
                ("Operations", "operations"),
                ("HR", "hr")
            ],
            "required": True,
            "style": "compact"
        },
        {
            "type": "choice",
            "id": "accessLevel",
            "label": "Required Access Level",
            "choices": [
                ("Standard User", "standard"),
                ("Power User (local admin)", "power"),
                ("Developer (Azure, GitHub)", "developer")
            ],
            "required": True,
            "style": "expanded"
        },
        {
            "type": "text",
            "id": "specialRequirements",
            "label": "Special Software Requirements",
            "placeholder": "List any specific software licenses needed...",
            "multiline": True,
            "required": False
        }
    ]
)

show_card(onboarding_card, "IT Onboarding Data Collection Card")
validate_and_report(onboarding_card, "Onboarding Card")

## 6. Generating JSON for Power Automate

When you pass a card to the Power Automate **Post adaptive card and wait for a response** action, the **Message** field expects a JSON string — not a Python dict. Use `json.dumps()` to produce the string. This section shows how to convert and inspect the output you would paste into Power Automate.

In [ ]:
# Convert any card dict to a JSON string ready for Power Automate.
# ensure_ascii=False preserves non-ASCII characters (accents, non-Latin scripts).
# indent=None produces compact JSON (no whitespace) — smaller payload for API calls.
# Use indent=2 during development when you need to read the output.

def card_to_json(card: dict, compact: bool = False) -> str:
    """
    Serialize an Adaptive Card dict to a JSON string.

    Parameters
    ----------
    card : dict
        Adaptive Card dictionary
    compact : bool
        If True, produces compact JSON (no whitespace) for use in API calls.
        If False, produces indented JSON for human readability.

    Returns
    -------
    str
        JSON string to paste into Power Automate's Message field
    """
    indent = None if compact else 2
    return json.dumps(card, indent=indent, ensure_ascii=False)


# Show the compact JSON you would paste into Power Automate
compact_json = card_to_json(expense_card, compact=True)

print("=== Compact JSON for Power Automate ===")
print(f"Character count: {len(compact_json)}")
print()
# Print first 300 chars to show the structure without flooding the output
print(compact_json[:300] + " ...")
print()

# Demonstrate round-trip: parse the JSON back to a dict and validate
reparsed = json.loads(compact_json)
validate_and_report(reparsed, "Round-trip validation")

## 7. Exercises

Apply what you have built to create cards for specific business scenarios. Each exercise has self-check tests that run automatically.

### Exercise 7.1: Custom Approval Card with Delegation Option

**Task:** Build an approval card for a contract sign-off that includes three decision options: `Approve`, `Reject`, and `Delegate`. When the approver selects Delegate, they should also be able to enter the delegate's email address in a text field.

The card must:
- Have a title: `"Contract Sign-Off Required"`
- Include facts: Contract ID, Value, Counterparty, Expiry Date
- Have a decision ChoiceSet with id `"decision"` and three choices: Approve / Reject / Delegate
- Have a text input with id `"delegateTo"` for the delegate email address
- Have a text input with id `"comments"` for comments
- Pass structural validation

**Hint:** You can use `build_approval_card` with custom `decision_choices`, then add the delegate field manually, or build from scratch using the element builders.

In [ ]:
# YOUR CODE HERE
# ---------------
# Build the contract sign-off card.
# Store the result in the variable: contract_card

contract_card = None  # Replace with your implementation


# Show your card (uncomment when ready)
# if contract_card:
#     show_card(contract_card, "Contract Sign-Off Card")

In [ ]:
# AUTO-GRADED TESTS — do not modify
# ----------------------------------

def test_exercise_7_1():
    assert contract_card is not None, \
        "contract_card is None. Build the card and assign it to the variable."

    assert isinstance(contract_card, dict), \
        "contract_card must be a dict, not a JSON string. Use make_card() or build_approval_card()."

    # Top-level structure
    assert contract_card.get("type") == "AdaptiveCard", \
        "Card 'type' must be 'AdaptiveCard'"

    assert contract_card.get("version") in SUPPORTED_VERSIONS, \
        f"Card version must be one of {sorted(SUPPORTED_VERSIONS)}"

    # Collect all input ids recursively
    all_ids = _collect_input_ids(contract_card.get("body", []))

    assert "decision" in all_ids, \
        "Card must contain an input with id='decision' for the choice set"

    assert "delegateTo" in all_ids, \
        "Card must contain an input with id='delegateTo' for the delegate email"

    assert "comments" in all_ids, \
        "Card must contain an input with id='comments'"

    # Verify the decision field has 3 choices
    def find_element_by_id(elements, target_id):
        for elem in elements:
            if not isinstance(elem, dict):
                continue
            if elem.get("id") == target_id:
                return elem
            if "items" in elem:
                found = find_element_by_id(elem["items"], target_id)
                if found:
                    return found
        return None

    decision_elem = find_element_by_id(contract_card.get("body", []), "decision")
    assert decision_elem is not None, "Could not find the 'decision' input element"

    choices = decision_elem.get("choices", [])
    assert len(choices) == 3, \
        f"'decision' must have exactly 3 choices (Approve, Reject, Delegate), found {len(choices)}"

    choice_values = [c.get("value", "").lower() for c in choices]
    assert any("delegate" in v for v in choice_values), \
        "One of the choices must have 'delegate' in its value string"

    # Structural validation
    errors = validate_card(contract_card)
    assert len(errors) == 0, \
        f"Card failed structural validation with {len(errors)} error(s):\n" + "\n".join(f"  - {e}" for e in errors)

    print("[PASS] Exercise 7.1: Contract sign-off card is valid with all required fields")


test_exercise_7_1()

### Exercise 7.2: Status Card for Rejection

**Task:** Using `build_status_card`, create a rejection notification card.

The card must:
- Have status text `"REJECTED"` with color `"Attention"`
- Include at least 3 facts in the summary FactSet
- Include a non-empty message explaining the rejection
- Pass structural validation

**Hint:** Use `build_status_card` with `status_color="Attention"` for the red color used in Teams for errors.

In [ ]:
# YOUR CODE HERE
# ---------------
# Build a rejection status card.
# Store the result in the variable: rejection_card

rejection_card = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS — do not modify
# ----------------------------------

def test_exercise_7_2():
    assert rejection_card is not None, \
        "rejection_card is None. Build the card using build_status_card()."

    assert isinstance(rejection_card, dict), \
        "rejection_card must be a dict"

    body = rejection_card.get("body", [])
    assert len(body) >= 3, \
        "Card body should have at least 3 elements: title, status, facts (and optionally a message)"

    # Find a TextBlock with color=Attention (the status indicator)
    attention_blocks = [
        elem for elem in body
        if isinstance(elem, dict)
        and elem.get("type") == "TextBlock"
        and elem.get("color") == "Attention"
    ]
    assert len(attention_blocks) >= 1, \
        "Card must contain a TextBlock with color='Attention' for the rejection status"

    # Status text must contain REJECTED
    status_texts = [b.get("text", "") for b in attention_blocks]
    assert any("REJECT" in t.upper() for t in status_texts), \
        f"Status TextBlock must contain 'REJECTED'. Found: {status_texts}"

    # Must have a FactSet with at least 3 facts
    fact_sets = [elem for elem in body if isinstance(elem, dict) and elem.get("type") == "FactSet"]
    assert len(fact_sets) >= 1, "Card must contain at least one FactSet"
    assert len(fact_sets[0].get("facts", [])) >= 3, \
        "FactSet must contain at least 3 facts"

    # Must have a non-empty message (look for Container with emphasis style)
    def has_emphasis_container_with_text(elements):
        for elem in elements:
            if not isinstance(elem, dict):
                continue
            if elem.get("type") == "Container" and elem.get("style") == "emphasis":
                # Check that it has at least one TextBlock with non-empty text
                inner = elem.get("items", [])
                for inner_elem in inner:
                    if isinstance(inner_elem, dict) and inner_elem.get("type") == "TextBlock":
                        if inner_elem.get("text", "").strip():
                            return True
        return False

    assert has_emphasis_container_with_text(body), \
        "Card must include a message in an emphasis Container explaining the rejection"

    # Structural validation
    errors = validate_card(rejection_card)
    assert len(errors) == 0, \
        f"Card failed structural validation:\n" + "\n".join(f"  - {e}" for e in errors)

    print("[PASS] Exercise 7.2: Rejection status card is valid with correct structure")


test_exercise_7_2()

### Exercise 7.3: Catch the Validation Errors

**Task:** The `broken_card` below has three structural errors. Run the validator to find them, then fix the card so it passes validation.

**Do not change the test — fix the card.**

In [ ]:
# This card has three structural errors.
# Run the validator first to see what they are, then fix the card.

broken_card = {
    "type": "AdaptiveCard",
    "version": "2.0",   # Error 1: unsupported version for Teams
    "body": [
        {
            "type": "TextBlock",
            "text": "Please review"
        },
        {
            "type": "Input.ChoiceSet",
                             # Error 2: missing 'id' field on this input
            "label": "Decision",
            "choices": [
                {"title": "Approve", "value": "approve"},
                {"title": "Reject", "value": "reject"}
            ]
        },
        {
            "type": "Input.Text",
            "id": "comments",
            "label": "Comments"
        },
        {
            "type": "Input.Text",
            "id": "comments",  # Error 3: duplicate id — same as the input above
            "label": "Additional Notes"
        }
    ],
    "actions": [
        {"type": "Action.Submit", "title": "Submit"}
    ]
}

# Step 1: Run the validator to see the errors
print("=== Validation report on broken_card ===")
errors = validate_card(broken_card)
for e in errors:
    print(f"  - {e}")
print()

# Step 2: Fix the card by editing the broken_card dict below
# YOUR CODE HERE — modify broken_card to fix all three errors
# ...

In [ ]:
# AUTO-GRADED TESTS — do not modify
# ----------------------------------

def test_exercise_7_3():
    errors = validate_card(broken_card)
    assert len(errors) == 0, (
        f"broken_card still has {len(errors)} error(s):\n"
        + "\n".join(f"  - {e}" for e in errors)
        + "\n\nFix all three errors in the broken_card dict above."
    )

    # Confirm the card still has both inputs with distinct ids
    all_ids = _collect_input_ids(broken_card.get("body", []))
    assert len(all_ids) == len(set(all_ids)), \
        "All input ids must be unique after fixing"

    assert len(all_ids) >= 2, \
        "Card must still have at least 2 input fields after fixing"

    print("[PASS] Exercise 7.3: broken_card is now valid — all three errors fixed")


test_exercise_7_3()

## Summary

### Key Takeaways

1. **Adaptive Cards are JSON dictionaries** — `type`, `version`, `body`, `actions` at the top level
2. **Python element builders** make cards readable, reusable, and consistent
3. **Always validate** before deploying — structural errors only surface at runtime in Power Automate
4. **Input `id` values** are your keys for reading responses — make them unique and descriptive
5. **Three card patterns**: approval (interactive), status (informational), data collection (structured input)
6. **`json.dumps(card)`** produces the string you paste into Power Automate's Message field

### What's Next

In Exercise 01 (exercises/01_approval_flow_exercise.py), you will apply the approval patterns from both guides to design complete flow logic for business scenarios — without a Power Automate environment. This trains the design thinking that drives good flow architecture.

### Additional Resources

- [Adaptive Cards Designer (interactive preview)](https://adaptivecards.io/designer/) — validate card JSON visually
- [Adaptive Cards Schema Explorer](https://adaptivecards.io/explorer/) — all element types and properties
- [Microsoft Docs: Post Adaptive Card in Teams](https://learn.microsoft.com/en-us/power-automate/adaptive-cards/create-adaptive-cards) — Power Automate integration reference
- [Adaptive Cards samples gallery](https://adaptivecards.io/samples/) — ready-made card layouts to adapt